In [1]:
import gzip
import itertools
import math
import os
import time
import tracemalloc
from collections import defaultdict
from fileReader import stream_dataset


class FPNode:
    def __init__(self, item, count, parent):
        self.item = item          
        self.count = count        
        self.parent = parent      
        self.children = {}        
        self.node_link = None     

In [2]:
class FPGrowth:
    def __init__(self, min_support=0.5, store_patterns=False):
        self.min_support = min_support
        self.min_support_count = 0
        self.total_transactions = 0
        self.store_patterns = store_patterns
        self.frequent_itemsets = {}
        self.total_frequent_count = 0

    def fit(self, filepath):
        """Builds the global FP-Tree in 2 scans and recursively mines all frequent itemsets."""
        self.frequent_itemsets.clear()
        self.total_frequent_count = 0

        # SCAN 1: Count 1-item frequencies & total transactions
        item_counts = defaultdict(int)
        tx_count = 0
        for tx in stream_dataset(filepath):
            tx_count += 1
            for item in tx:
                item_counts[item] += 1

        self.total_transactions = tx_count
        if tx_count == 0:
            return 0

        self.min_support_count = math.ceil(self.min_support * self.total_transactions)

        freq_items = {
            item: count
            for item, count in item_counts.items()
            if count >= self.min_support_count
        }
        if not freq_items:
            return 0

        # Sort by count descending (-x[1]), tie-break by item ID ascending (x[0])
        sorted_items = sorted(freq_items.items(), key=lambda x: (-x[1], x[0]))


        # Header Table: {item: [total_count, head_node, tail_node]}
        header_table = {item: [count, None, None] for item, count in sorted_items}
        
        rank = {item: idx for idx, (item, _) in enumerate(sorted_items)}

        # SCAN 2: Build the global FP-Tree
        root = FPNode(item=None, count=0, parent=None)
        for tx in stream_dataset(filepath):
            ordered_tx = [item for item in tx if item in freq_items]
            if ordered_tx:
                ordered_tx.sort(key=rank.__getitem__)
                self._insert_tree(ordered_tx, root, header_table, count=1)

        self._mine_tree(root, header_table, prefix=())
        return self.total_frequent_count, self.total_transactions, self.min_support_count

    def _insert_tree(self, items, root, header_table, count):
        """Iteratively inserts a sorted transaction into the FP-Tree using O(1) tail pointers."""
        curr = root
        for item in items:
            children = curr.children
            if item in children:
                child = children[item]
                child.count += count
            else:
                child = FPNode(item=item, count=count, parent=curr)
                children[item] = child

                # O(1) append to horizontal node-link chain: [total_count, head, tail]
                entry = header_table[item]
                if entry[1] is None:
                    entry[1] = child
                    entry[2] = child
                else:
                    entry[2].node_link = child
                    entry[2] = child
            curr = child

    def _is_single_path(self, node):
        # Checks if the subtree below `node` is a single unbranched chain.
        curr = node
        while curr.children:
            if len(curr.children) > 1:
                return False
            curr = next(iter(curr.children.values()))
        return True

    def _record_pattern(self, pattern_tuple, support):
        self.total_frequent_count += 1
        if self.store_patterns:
            self.frequent_itemsets[pattern_tuple] = support

    def _mine_tree(self, root, header_table, prefix):
        """Recursively mines frequent itemsets with Single-Path Optimization."""
        if self._is_single_path(root):
            path_nodes = []
            curr = root
            while curr.children:
                curr = next(iter(curr.children.values()))
                path_nodes.append((curr.item, curr.count))

            n = len(path_nodes)
            if not self.store_patterns:
                self.total_frequent_count += (1 << n) - 1
            else:
                for r in range(1, n + 1):
                    for combo in itertools.combinations(path_nodes, r):
                        items = tuple(sorted(prefix + tuple(x[0] for x in combo)))
                        min_sup = min(x[1] for x in combo)
                        self._record_pattern(items, min_sup)
            return

        # Bottom-up mining: sort by count ascending (x[1][0]), tie-break by item ID descending (-x[0])
        bottom_up_items = sorted(header_table.items(), key=lambda x: (x[1][0], -x[0]))

        for item, (support, head_node, _) in bottom_up_items:
            new_prefix = prefix + (item,)
            if self.store_patterns:
                new_prefix = tuple(sorted(new_prefix))
            self._record_pattern(new_prefix, support)

            #Extract Conditional Pattern Base
            cond_patterns = []
            curr = head_node
            while curr is not None:
                path = []
                p = curr.parent
                while p is not None and p.item is not None:
                    path.append(p.item)
                    p = p.parent
                if path:
                    path.reverse()
                    cond_patterns.append((path, curr.count))
                curr = curr.node_link

            if not cond_patterns:
                continue

            # Count conditional item frequencies
            cond_counts = defaultdict(int)
            for path, cnt in cond_patterns:
                for p_item in path:
                    cond_counts[p_item] += cnt

            freq_cond = {
                p_item: cnt
                for p_item, cnt in cond_counts.items()
                if cnt >= self.min_support_count
            }
            if not freq_cond:
                continue

            # Build Conditional FP-Tree
            sorted_cond = sorted(freq_cond.items(), key=lambda x: (-x[1], x[0]))
            cond_header = {p_item: [cnt, None, None] for p_item, cnt in sorted_cond}
            cond_rank = {p_item: idx for idx, (p_item, _) in enumerate(sorted_cond)}

            cond_root = FPNode(item=None, count=0, parent=None)
            for path, cnt in cond_patterns:
                ordered_path = [p_item for p_item in path if p_item in freq_cond]
                if ordered_path:
                    ordered_path.sort(key=cond_rank.__getitem__)
                    self._insert_tree(ordered_path, cond_root, cond_header, cnt)

            # 4. Recurse into Conditional FP-Tree
            if cond_header:
                self._mine_tree(cond_root, cond_header, new_prefix)


In [3]:
def run_benchmark(dataset_name, filepath, support_thresholds):
    print(f"\n{'='*100}")
    print(f"Dataset: {dataset_name}")
    print(f"{'MinSup (%)':<12} | {'Total Transactions':<15} | {'Min Support Count':<15} | {'Freq Itemsets':<15} | {'Runtime (s)':<14} | {'Peak Memory (MB)':<16}")
    print(f"{'-'*100}")

    for sup in support_thresholds:
        tracemalloc.start()

        start_time = time.perf_counter()

        model = FPGrowth(min_support=sup, store_patterns=False)
        freq_count, total_transactions, min_support_count = model.fit(filepath)

        elapsed = time.perf_counter() - start_time
        _, peak_bytes = tracemalloc.get_traced_memory()
        
        tracemalloc.stop()

        peak_mb = peak_bytes / (1024 * 1024)
        print(f"{sup*100:<11.2f}% | {total_transactions:<18.2f} | {min_support_count:<18.1f} | {freq_count:<15} | {elapsed:<14.4f} | {peak_mb:<16.4f}")
        

In [6]:
if __name__ == "__main__":
    dense_configs = [
        ("Chess --> Dense Dataset", "datasets/chess.dat.gz", [0.70, 0.75, 0.80, 0.85, 0.90]),
        ("Mushroom --> Dense Dataset", "datasets/mushroom.dat.gz", [0.35, 0.40, 0.45, 0.50, 0.60]),
    ]
    sparse_configs = [
        ("Retail --> Sparse Dataset", "datasets/retail.dat.gz", [0.005, 0.01, 0.015, 0.02, 0.03]),
        ("kosarak --> Sparse Dataset", "datasets/kosarak.dat.gz", [0.005, 0.01, 0.015, 0.02, 0.025]),
    ]

    for name, path, thresholds in dense_configs + sparse_configs:
        run_benchmark(name, path, thresholds)



Dataset: Chess --> Dense Dataset
MinSup (%)   | Total Transactions | Min Support Count | Freq Itemsets   | Runtime (s)    | Peak Memory (MB)
----------------------------------------------------------------------------------------------------
70.00      % | 3196.00            | 2238.0             | 48731           | 1.5068         | 16.8403         
75.00      % | 3196.00            | 2397.0             | 20993           | 0.6509         | 3.0397          
80.00      % | 3196.00            | 2557.0             | 8227            | 0.3396         | 4.6122          
85.00      % | 3196.00            | 2717.0             | 2669            | 0.2206         | 1.6327          
90.00      % | 3196.00            | 2877.0             | 622             | 0.1753         | 0.4177          

Dataset: Mushroom --> Dense Dataset
MinSup (%)   | Total Transactions | Min Support Count | Freq Itemsets   | Runtime (s)    | Peak Memory (MB)
-------------------------------------------------------------------